# 采样性能对比

我们唯一称得上创新点的小玩意，虽然在整个推理流程中比例不大，但理论上应该是快了一些

## 1. 初始化

加载模型，完成prefill阶段，得到初始logits

In [1]:
from helper import load_from_cache, prefill

base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

prompt = "python is"
input_ids = tokenizer.encode(prompt, return_tensors='jax')
logits, _ = prefill(model, params, input_ids)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


## 2. 定义采样函数

- 基线：什么都不干，输入logits，返回logits中的第一个，作为基础通信量和时间的参考
- 贪心采样：取原始值最高的logit输出，无需softmax等计算，V-1次比较
- topk采样：取原始值最高的k个，在这k个中取样，O(kV)
- minp采样：找到最高概率pmax，设置最低概率pmin=minp*pmax，在概率pmin-pmax的logits中采样
- 老代码：跑出来非常糟糕结果的老采样代码

In [2]:
import jax
import jax.numpy as jnp

# 基线，只测时间
def sampler_baseline(logits):
    return [0]

# 贪心采样
def sampler_greedy(logits):
    return jnp.argmax(logits, axis=-1)

# top k
def sampler_top_k(logits, top_k = 40, seed = 42):

    key = jax.random.PRNGKey(seed)

    # 1. 找到第 k 大的 logit 值（阈值）
    values, _ = jax.lax.top_k(logits, k=top_k)
    kth = values[:, -1:]                     # 形状 (batch, 1)

    # 2. 掩码：小于阈值的设为 -1e9（或 -inf）
    masked_logits = jnp.where(logits < kth, -1e9, logits)

    # 3. 直接采样（categorical 内部会做 softmax）
    return jax.random.categorical(key, masked_logits, axis=-1)

# min p，变形优化版本
def sampler_min_p(logits, min_p =0.1, seed = 42):

    key = jax.random.PRNGKey(seed)

    max_logit = jnp.max(logits, axis=-1, keepdims=True)
    threshold = jnp.log(min_p) + max_logit # math hack
    mask = logits >= threshold
    filtered_logits = jnp.where(mask, logits, -1e10)

    # 随机采样
    key, subkey = jax.random.split(key)
    return jax.random.categorical(subkey, filtered_logits, axis=-1)

# 老代码
def sampler_legacy(logits, sample: bool = True, top_k: int = 40, seed: int = 42):
    key = jax.random.PRNGKey(seed)
    if top_k is not None:
        values, _ = jax.lax.top_k(logits, k=top_k)
        kth_values = values[:, -1:]
        logits = jnp.where(logits < kth_values, -1e9, logits)

    probs = jax.nn.softmax(logits, axis=-1)

    if sample:
        key, subkey = jax.random.split(key)
        next_id = jax.random.categorical(subkey, jnp.log(probs + 1e-9), axis=-1)
    else:
        next_id = jnp.argmax(probs, axis=-1)

    return next_id


## 3. 明文测试

In [3]:
next_id = sampler_baseline(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_top_k(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_min_p(logits)
print(prompt, tokenizer.decode(next_id), sep="")

next_id =sampler_legacy(logits)
print(prompt, tokenizer.decode(next_id), sep="")

python is<|endoftext|>
python is not
python is in
python is in


## 4. spu测试

In [4]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS
# emulator = emulation.Emulator("3pc.json", mode)  # aby3
emulator = emulation.Emulator("2pc.json", mode)  # cheetah

emulator.up()

s_logits = emulator.seal(logits)

[2026-04-09 17:14:41,594]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-09 17:14:42,005] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61321
[2026-04-09 17:14:42,022] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61320
[2026-04-09 17:14:43,692] [ForkServerProcess-2] Run : builtin_spu_init at node:1
[2026-04-09 17:14:43,692] [ForkServerProcess-1] Run : builtin_spu_init at node:0
I0409 17:14:43.698610 14855     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61331.
W0409 17:14:43.698630 14855     0 external/brpc~/src/brpc/server.cpp:1201] Builtin services are disabled according to ServerOptions.has_builtin_services
I0409 17:14:43.698801 14854     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61330.
W0409 17:14:43.698813 14854     0 external/brpc~/src/brpc/server.cpp:1201] Builtin services are disabl

### 4.1 baseline

In [5]:
next_id = emulator.run(sampler_baseline)(s_logits)

[2026-04-09 17:14:43.807] [info] [thread_pool.cc:30] Create a fixed thread pool with size 19
[2026-04-09 17:14:43.831] [info] [api.cc:172] [Profiling] SPU execution sampler_baseline completed, input processing took 6.41e-07s, execution took 0.00050291s, output processing took 1.772e-06s, total time 0.000505323s.
[2026-04-09 17:14:43.831] [info] [api.cc:220] HLO profiling: total time 1.3818e-05
[2026-04-09 17:14:43.831] [info] [api.cc:223] - pphlo.constant, executed 1 times, duration 1.3818e-05s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-09 17:14:43.831] [info] [api.cc:220] HAL profiling: total time 0
[2026-04-09 17:14:43.831] [info] [api.cc:220] MPC profiling: total time 0
[2026-04-09 17:14:43.831] [info] [api.cc:233] Link details: total send bytes 0, recv bytes 0, send actions 0, recv actions 0


[2026-04-09 17:14:43,806] [ForkServerProcess-1] Run : make_shares at node:0
[2026-04-09 17:14:43,816] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:14:43,830] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-09 17:14:43,830] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-09 17:14:43,832] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-09 17:14:43,842] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:14:43,844] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-09 17:14:43,845] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1


In [6]:
print(prompt, tokenizer.decode(next_id), sep="")

python is<|endoftext|>


### 4.2 greedy

In [7]:
next_id = emulator.run(sampler_greedy)(s_logits)

[2026-04-09 17:14:43,912] [ForkServerProcess-1] Run : make_shares at node:0
[2026-04-09 17:14:43,921] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:14:43,942] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-09 17:14:43,942] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-09 17:14:43,944] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0


[2026-04-09 17:14:43.950] [info] [thread_pool.cc:30] Create a fixed thread pool with size 19
[2026-04-09 17:14:51.538] [info] [api.cc:172] [Profiling] SPU execution sampler_greedy completed, input processing took 2.092e-06s, execution took 7.59496204s, output processing took 1.789e-06s, total time 7.594965921s.
[2026-04-09 17:14:51.538] [info] [api.cc:220] HLO profiling: total time 7.594177633
[2026-04-09 17:14:51.538] [info] [api.cc:223] - pphlo.reduce, executed 1 times, duration 7.591321442s, send bytes 32461038 recv bytes 16371758, send actions 32006, recv actions 32006
[2026-04-09 17:14:51.538] [info] [api.cc:223] - pphlo.convert, executed 3 times, duration 0.002242225s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-09 17:14:51.538] [info] [api.cc:223] - pphlo.iota, executed 1 times, duration 0.000562435s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-09 17:14:51.538] [info] [api.cc:223] - pphlo.reshape, executed 1 times, duration 2.619e-05

[2026-04-09 17:14:51,541] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:14:51,544] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-09 17:14:51,546] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1


In [8]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 4.3 top k

In [9]:
next_id = emulator.run(sampler_top_k)(s_logits)

[2026-04-09 17:14:51,670] [ForkServerProcess-1] Run : make_shares at node:0
[2026-04-09 17:14:51,682] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:14:51,903] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-09 17:14:51,903] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-09 17:14:51,905] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0


[2026-04-09 17:14:52.235] [info] [cheetah_mul.cc:335] CheetahMul uses 7 modulus for 128 bit input over 128 bit ring
[2026-04-09 17:14:52.255] [info] [cheetah_mul.cc:335] CheetahMul uses 7 modulus for 128 bit input over 128 bit ring
[2026-04-09 17:15:03.549] [info] [api.cc:172] [Profiling] SPU execution sampler_top_k completed, input processing took 1.75e-06s, execution took 11.645587958s, output processing took 1.142e-06s, total time 11.64559085s.
[2026-04-09 17:15:03.549] [info] [api.cc:220] HLO profiling: total time 11.642931912
[2026-04-09 17:15:03.549] [info] [api.cc:223] - pphlo.custom_call: mhlo.topk, executed 1 times, duration 8.358814801s, send bytes 400898705 recv bytes 390865139, send actions 19248, recv actions 19248
[2026-04-09 17:15:03.549] [info] [api.cc:223] - pphlo.reduce, executed 1 times, duration 2.986869946s, send bytes 23420070 recv bytes 6798272, send actions 5838, recv actions 3276
[2026-04-09 17:15:03.549] [info] [api.cc:223] - pphlo.less, executed 2 times, dura

[2026-04-09 17:15:03,553] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:15:03,555] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-09 17:15:03,557] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1


In [10]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 4.4 min p

In [11]:
next_id = emulator.run(sampler_min_p)(s_logits)

[2026-04-09 17:15:03,681] [ForkServerProcess-1] Run : make_shares at node:0
[2026-04-09 17:15:03,690] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:15:03,843] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-09 17:15:03,844] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-09 17:15:03,845] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0


[2026-04-09 17:15:07.753] [info] [api.cc:172] [Profiling] SPU execution sampler_min_p completed, input processing took 2.287e-06s, execution took 3.909046629s, output processing took 1.054e-06s, total time 3.90904997s.
[2026-04-09 17:15:07.754] [info] [api.cc:220] HLO profiling: total time 3.9056692750000006
[2026-04-09 17:15:07.754] [info] [api.cc:223] - pphlo.reduce, executed 2 times, duration 3.616165212s, send bytes 31497321 recv bytes 8978016, send actions 8169, recv actions 4326
[2026-04-09 17:15:07.754] [info] [api.cc:223] - pphlo.less, executed 2 times, duration 0.165091734s, send bytes 6991360 recv bytes 1360000, send actions 460, recv actions 460
[2026-04-09 17:15:07.754] [info] [api.cc:223] - pphlo.select, executed 2 times, duration 0.028765905s, send bytes 810880 recv bytes 810880, send actions 40, recv actions 40
[2026-04-09 17:15:07.754] [info] [api.cc:223] - pphlo.or, executed 41 times, duration 0.022281398s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[202

[2026-04-09 17:15:07,757] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:15:07,760] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-09 17:15:07,762] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1


In [12]:
print(prompt, tokenizer.decode(next_id), sep="")

python is not


### 4.5 legacy

In [13]:
next_id = emulator.run(sampler_legacy)(s_logits)

[2026-04-09 17:15:07,918] [ForkServerProcess-1] Run : make_shares at node:0
[2026-04-09 17:15:07,925] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:15:08,030] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-09 17:15:08,030] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-09 17:15:08,032] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0


[2026-04-09 17:15:43.812] [info] [api.cc:172] [Profiling] SPU execution sampler_legacy completed, input processing took 1.753e-06s, execution took 35.781408851s, output processing took 1.933e-06s, total time 35.781412537s.
[2026-04-09 17:15:43.812] [info] [api.cc:220] HLO profiling: total time 35.777672645
[2026-04-09 17:15:43.812] [info] [api.cc:223] - pphlo.log, executed 3 times, duration 9.40321886s, send bytes 207955426 recv bytes 124003534, send actions 24486, recv actions 10335
[2026-04-09 17:15:43.812] [info] [api.cc:223] - pphlo.divide, executed 1 times, duration 9.213834245s, send bytes 211047022 recv bytes 169251049, send actions 17540, recv actions 26487
[2026-04-09 17:15:43.812] [info] [api.cc:223] - pphlo.exponential, executed 1 times, duration 7.063087012s, send bytes 219189982 recv bytes 143063535, send actions 15226, recv actions 3657
[2026-04-09 17:15:43.812] [info] [api.cc:223] - pphlo.custom_call: mhlo.topk, executed 1 times, duration 5.096865203s, send bytes 3953411

[2026-04-09 17:15:43,816] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-09 17:15:43,818] [ForkServerProcess-1] RunR: builtin_fetch_object at node:0
[2026-04-09 17:15:43,819] [ForkServerProcess-2] RunR: builtin_fetch_object at node:1


In [14]:
print(prompt, tokenizer.decode(next_id), sep="")

python isF


## 5. cleanup

In [15]:
emulator.down()

[2026-04-09 17:15:43,975]-[INFO]-[emulation.py:120]: Shutdown multiprocess cluster...
